# CSI Fingerprint Analysis

Explore individual and mean CSI fingerprints already produced by the classical ML feature pipeline. This notebook deliberately adds **no post-extraction normalization, StandardScaler, distance metric, or model**.

## Imports

In [ ]:
import math
import re

import matplotlib.pyplot as plt
import pandas as pd
from IPython.display import display

from utils.config import (
    BANDS_TO_RUN,
    DATA_DIR,
    DEFAULT_OVERLAP_SIZE,
    DEFAULT_WINDOW_SIZE,
)
from utils.feature_pipeline import FEATURE_NAMES, METADATA_COLUMNS
from utils.ML.ml_pipeline import (
    get_cache_path,
    load_feature_dataframes,
    load_raw_csi_data,
)

## Project and pipeline configuration

These settings mirror the ML notebook. `load_feature_dataframes` reuses its preprocessing, feature extraction, and cache pipeline; this notebook does not implement any of those operations again.

In [ ]:
CALIBRATION_MODE = "rssi"  # none | packet_norm | rssi

CSV_PROCESSING_OPTIONS = {
    "max_workers": 1,
    "cache_dir": None,
    "use_cache": True,
    "force_reprocess": False,
    "min_rssi_dbm": -95.0,
    "calibration_eps": 1e-12,
}

MAGNITUDE_PROCESSING_OPTIONS = {
    "normalization": "empty_baseline",  # same preprocessing choice as ML.ipynb
    "epsilon": 1e-8,
}
NORMALIZATION_BASELINE_SCOPE = "per_session"

FEATURE_EXTRACTION_OPTIONS = {
    "window_size": DEFAULT_WINDOW_SIZE,
    "overlap_size": DEFAULT_OVERLAP_SIZE,
    "require_all_esps": False,
}

preproc_opts = dict(MAGNITUDE_PROCESSING_OPTIONS)
if preproc_opts.get("normalization") == "empty_baseline":
    preproc_opts["baseline_scope"] = NORMALIZATION_BASELINE_SCOPE
feat_opts = dict(FEATURE_EXTRACTION_OPTIONS)

print(f"Feature cache path: {get_cache_path(preproc_opts, feat_opts)}")

## Fingerprint analysis configuration

`WINDOW_INDEX` is the zero-based `window_idx` stored by the ML pipeline. Set `ESPS_TO_PLOT` to `"all"` or a tuple such as `("esp_01", "esp_11")`.

In [ ]:
POSITION = "B-8"
USER = "03"
TRIAL = "01"
SENSING_CONFIGURATION = "Fusion"  # 2.4 GHz | 5 GHz | Fusion
SCENARIO = None  # Optional; leave as None when position/user/trial identifies one recording

ANALYSIS_MODE = "individual"  # individual | mean
WINDOW_INDEX = 0
FEATURE_FAMILY = "mean"  # mean | std | variance | max | min | energy
ESPS_TO_PLOT = "all"

COMPARE_TRIALS = True
COMPARISON_TRIALS = ("01", "02")

DISPLAY_ROWS_PER_ESP = 8  # Set to None to display the complete organized vector
SHARE_Y_AXIS = True       # Useful for comparing absolute amplitudes between ESP panels
PLOT_COMBINED_ESPS = False  # Optional overlay; can be busy for Fusion

## Raw data loading

In [ ]:
magnitude_data, csv_diagnostics = load_raw_csi_data(
    DATA_DIR,
    calibration_mode=CALIBRATION_MODE,
    csv_options=CSV_PROCESSING_OPTIONS,
)
display(csv_diagnostics.head())

## Existing preprocessing and feature DataFrames

The next call is the same cached preprocessing/feature path used by `ML.ipynb`. The selected feature values below are consumed directly from these DataFrames—there is no scaling after extraction.

In [ ]:
feature_dataframes = load_feature_dataframes(
    magnitude_data,
    preproc_opts=preproc_opts,
    feat_opts=feat_opts,
    bands_to_run=BANDS_TO_RUN,
)

feature_inventory = pd.DataFrame(
    [
        {
            "sensing_configuration": band,
            "fingerprints": len(df),
            "feature_columns": len(df.columns.difference(METADATA_COLUMNS)),
        }
        for band, df in feature_dataframes.items()
    ]
)
display(feature_inventory)
del magnitude_data

## Selection and organization helpers

In [ ]:
FEATURE_COLUMN_RE = re.compile(
    r"^(esp_\d+)_(mean|std|variance|max|min|energy)_sc_(\d+)$"
)
FEATURE_FAMILY_ALIASES = {"maximum": "max", "minimum": "min"}
FEATURE_LABELS = {
    "mean": "Mean",
    "std": "Standard deviation",
    "variance": "Variance",
    "max": "Maximum",
    "min": "Minimum",
    "energy": "Energy",
}


def _two_digit(value):
    return str(value).removeprefix("user_").removeprefix("trial_").removesuffix(".0").zfill(2)


def _position(value):
    return str(value).removeprefix("location_").upper()


def canonical_feature_family(feature_family):
    family = FEATURE_FAMILY_ALIASES.get(str(feature_family).lower(), str(feature_family).lower())
    if family not in FEATURE_NAMES:
        raise ValueError(f"Unknown feature family {feature_family!r}; choose one of {FEATURE_NAMES}.")
    return family


def fingerprint_feature_columns(df):
    return [column for column in df.columns if FEATURE_COLUMN_RE.fullmatch(str(column))]


def select_recording_rows(df, *, position, user, trial, scenario=None):
    required = {"location", "user", "trial", "window_idx"}
    missing = required - set(df.columns)
    if missing:
        raise KeyError(f"Feature DataFrame is missing metadata columns: {sorted(missing)}")

    mask = (
        df["location"].map(_position).eq(_position(position))
        & df["user"].map(_two_digit).eq(_two_digit(user))
        & df["trial"].map(_two_digit).eq(_two_digit(trial))
    )
    if scenario is not None:
        if "scenario" not in df.columns:
            raise KeyError("SCENARIO was set, but the DataFrame has no 'scenario' column.")
        scenario_value = str(scenario).removeprefix("scenario_")
        mask &= df["scenario"].astype(str).str.removeprefix("scenario_").eq(scenario_value)

    rows = df.loc[mask].sort_values("window_idx").copy()
    if rows.empty:
        raise ValueError(
            f"No fingerprints for position={position}, user={_two_digit(user)}, "
            f"trial={_two_digit(trial)}."
        )
    if scenario is None and "scenario" in rows.columns and rows["scenario"].nunique() > 1:
        choices = sorted(rows["scenario"].astype(str).unique())
        raise ValueError(f"Selection matches multiple scenarios {choices}; set SCENARIO explicitly.")
    if rows["window_idx"].duplicated().any():
        raise ValueError("Selection contains duplicate window_idx values; refine SCENARIO.")
    return rows


def select_fingerprint(recording_rows, *, mode, window_index):
    mode = str(mode).lower()
    columns = fingerprint_feature_columns(recording_rows)
    if not columns:
        raise ValueError("No fingerprint feature columns were found.")

    if mode == "individual":
        match = recording_rows.loc[recording_rows["window_idx"].eq(window_index)]
        if len(match) != 1:
            available = recording_rows["window_idx"].astype(int).tolist()
            raise ValueError(f"window_idx={window_index} is unavailable; choose from {available}.")
        return match.iloc[0][columns].astype(float), 1
    if mode == "mean":
        return recording_rows[columns].mean(axis=0).astype(float), len(recording_rows)
    raise ValueError("ANALYSIS_MODE must be 'individual' or 'mean'.")


def organize_fingerprint(fingerprint):
    records = []
    for column, value in fingerprint.items():
        match = FEATURE_COLUMN_RE.fullmatch(str(column))
        if match is None:
            continue
        esp, family, subcarrier_text = match.groups()
        subcarrier_index = int(subcarrier_text)
        records.append(
            {
                "ESP": esp,
                "subcarrier_index": subcarrier_index,
                "subcarrier_number": subcarrier_index + 1,
                "feature_family": family,
                "value": float(value),
            }
        )
    long = pd.DataFrame.from_records(records)
    if long.empty:
        raise ValueError("The selected fingerprint contains no recognized feature values.")
    table = (
        long.pivot(
            index=["ESP", "subcarrier_index", "subcarrier_number"],
            columns="feature_family",
            values="value",
        )
        .reindex(columns=list(FEATURE_NAMES))
        .sort_index()
    )
    table.columns.name = None
    return table


def resolve_esps(table, requested_esps="all"):
    available = list(dict.fromkeys(table.index.get_level_values("ESP")))
    if requested_esps == "all":
        return available
    requested = [str(esp).lower() for esp in requested_esps]
    unknown = sorted(set(requested) - set(available))
    if unknown:
        raise ValueError(f"Requested ESPs are unavailable in this fingerprint: {unknown}")
    return [esp for esp in available if esp in requested]

## Reusable plotting helpers

Faceted plots retain a separate subcarrier axis for every ESP. Shared y-axes compare absolute values without normalizing them; disable `SHARE_Y_AXIS` when one ESP dominates the scale.

In [ ]:
def plot_fingerprint_facets(
    fingerprints,
    *,
    feature_family,
    esps="all",
    share_y=True,
    title=None,
):
    """Plot one or more organized fingerprints in a separate panel per ESP."""
    family = canonical_feature_family(feature_family)
    first_table = next(iter(fingerprints.values()))
    selected_esps = resolve_esps(first_table, esps)
    ncols = min(3, len(selected_esps))
    nrows = math.ceil(len(selected_esps) / ncols)
    fig, axes = plt.subplots(
        nrows,
        ncols,
        figsize=(5.2 * ncols, 3.3 * nrows),
        sharey=share_y,
        squeeze=False,
    )

    for ax, esp in zip(axes.flat, selected_esps):
        plotted = False
        for label, table in fingerprints.items():
            if esp not in table.index.get_level_values("ESP"):
                continue
            esp_table = table.xs(esp, level="ESP")
            values = esp_table[family].dropna()
            if values.empty:
                continue
            subcarrier_numbers = values.index.get_level_values("subcarrier_number")
            ax.plot(subcarrier_numbers, values.to_numpy(), linewidth=1.6, label=label)
            plotted = True
        ax.set_title(esp.upper())
        ax.set_xlabel("Retained subcarrier number (1-based)")
        ax.set_ylabel(FEATURE_LABELS[family])
        ax.grid(alpha=0.25)
        if plotted and len(fingerprints) > 1:
            ax.legend()

    for ax in axes.flat[len(selected_esps):]:
        ax.set_visible(False)
    fig.suptitle(title or f"{FEATURE_LABELS[family]} across retained subcarriers", y=1.01)
    fig.tight_layout()
    return fig, axes


def plot_combined_esps(table, *, feature_family, esps="all", title=None):
    """Optionally overlay ESP curves while preserving each ESP's native subcarrier count."""
    family = canonical_feature_family(feature_family)
    selected_esps = resolve_esps(table, esps)
    fig, ax = plt.subplots(figsize=(12, 5))
    for esp in selected_esps:
        esp_table = table.xs(esp, level="ESP")
        values = esp_table[family].dropna()
        if values.empty:
            continue
        x = values.index.get_level_values("subcarrier_number")
        ax.plot(x, values.to_numpy(), linewidth=1.2, alpha=0.85, label=esp.upper())
    ax.set(
        title=title or f"ESP comparison: {FEATURE_LABELS[family]}",
        xlabel="Retained subcarrier number (1-based)",
        ylabel=FEATURE_LABELS[family],
    )
    ax.grid(alpha=0.25)
    ax.legend(ncol=min(5, max(1, len(selected_esps))), fontsize=8)
    fig.tight_layout()
    return fig, ax

## Select an individual or mean fingerprint

In [ ]:
if SENSING_CONFIGURATION not in feature_dataframes:
    raise KeyError(
        f"Unknown SENSING_CONFIGURATION {SENSING_CONFIGURATION!r}; "
        f"choose from {tuple(feature_dataframes)}."
    )

active_df = feature_dataframes[SENSING_CONFIGURATION]
recording_rows = select_recording_rows(
    active_df,
    position=POSITION,
    user=USER,
    trial=TRIAL,
    scenario=SCENARIO,
)
selected_fingerprint, fingerprints_used = select_fingerprint(
    recording_rows,
    mode=ANALYSIS_MODE,
    window_index=WINDOW_INDEX,
)
selected_fingerprint_table = organize_fingerprint(selected_fingerprint)
selected_esps = resolve_esps(selected_fingerprint_table, ESPS_TO_PLOT)

selection_summary = pd.Series(
    {
        "position": _position(POSITION),
        "user": _two_digit(USER),
        "trial": _two_digit(TRIAL),
        "sensing_configuration": SENSING_CONFIGURATION,
        "analysis_mode": ANALYSIS_MODE,
        "window_idx": WINDOW_INDEX if ANALYSIS_MODE == "individual" else "all",
        "available_individual_fingerprints": len(recording_rows),
        "fingerprints_used_for_selected_vector": fingerprints_used,
        "feature_values_in_vector": int(selected_fingerprint.notna().sum()),
        "ESPs_in_vector": selected_fingerprint_table.index.get_level_values("ESP").nunique(),
    },
    name="selection",
)
display(selection_summary.to_frame())

## Organized feature values

The table keeps the exact ESP, zero-based pipeline subcarrier index, one-based display number, and six statistical families together. `selected_fingerprint_table` always contains the full organized vector even when only a preview is displayed.

In [ ]:
if DISPLAY_ROWS_PER_ESP is None:
    vector_preview = selected_fingerprint_table
else:
    vector_preview = selected_fingerprint_table.groupby(level="ESP", sort=False).head(
        DISPLAY_ROWS_PER_ESP
    )
    print(
        f"Showing the first {DISPLAY_ROWS_PER_ESP} subcarriers per ESP; "
        "set DISPLAY_ROWS_PER_ESP=None for the complete table."
    )
display(vector_preview)

selected_family = canonical_feature_family(FEATURE_FAMILY)
selected_family_table = (
    selected_fingerprint_table[selected_family]
    .unstack("ESP")
    .droplevel("subcarrier_index")
    .reindex(columns=selected_esps)
)
selected_family_table.index.name = "subcarrier_number"
print(f"{FEATURE_LABELS[selected_family]} values by retained subcarrier and ESP:")
display(selected_family_table)

## Selected feature family plots

In [ ]:
mode_label = (
    f"individual window_idx={WINDOW_INDEX}"
    if ANALYSIS_MODE == "individual"
    else f"mean of {fingerprints_used} fingerprints"
)
plot_fingerprint_facets(
    {mode_label: selected_fingerprint_table},
    feature_family=FEATURE_FAMILY,
    esps=ESPS_TO_PLOT,
    share_y=SHARE_Y_AXIS,
    title=(
        f"{POSITION} / User {USER} / Trial {TRIAL} / {SENSING_CONFIGURATION} — "
        f"{mode_label}"
    ),
)
plt.show()

if PLOT_COMBINED_ESPS:
    plot_combined_esps(
        selected_fingerprint_table,
        feature_family=FEATURE_FAMILY,
        esps=ESPS_TO_PLOT,
        title=f"{POSITION} / User {USER} / Trial {TRIAL} — ESP overlay",
    )
    plt.show()

## Trial 01 versus Trial 02 mean fingerprints

This comparison always calculates one column-wise mean from the existing individual fingerprint rows for each trial. No additional scaling is applied.

In [ ]:
if COMPARE_TRIALS:
    trial_tables = {}
    trial_counts = {}
    missing_trials = []

    for comparison_trial in COMPARISON_TRIALS:
        try:
            trial_rows = select_recording_rows(
                active_df,
                position=POSITION,
                user=USER,
                trial=comparison_trial,
                scenario=SCENARIO,
            )
        except ValueError as error:
            if str(error).startswith("No fingerprints for"):
                missing_trials.append(_two_digit(comparison_trial))
                continue
            raise

        mean_fingerprint, count = select_fingerprint(
            trial_rows,
            mode="mean",
            window_index=WINDOW_INDEX,
        )
        trial_label = f"Trial {_two_digit(comparison_trial)} (n={count})"
        trial_tables[trial_label] = organize_fingerprint(mean_fingerprint)
        trial_counts[_two_digit(comparison_trial)] = count

    comparison_inventory = pd.DataFrame(
        {
            "trial": list(trial_counts),
            "individual_fingerprints_used": list(trial_counts.values()),
        }
    )
    display(comparison_inventory)

    if missing_trials:
        print(
            "Trial comparison skipped because both trials are required. "
            f"Missing: {', '.join(missing_trials)}"
        )
    elif len(trial_tables) == 2:
        plot_fingerprint_facets(
            trial_tables,
            feature_family=FEATURE_FAMILY,
            esps=ESPS_TO_PLOT,
            share_y=SHARE_Y_AXIS,
            title=(
                f"Mean fingerprint comparison — {POSITION} / User {USER} / "
                f"{SENSING_CONFIGURATION}"
            ),
        )
        plt.show()
else:
    print("Trial comparison disabled. Set COMPARE_TRIALS=True to enable it.")